## Extracción de datos de empresas constituidas desde la API del INE

### Objetivo
Este notebook extrae y estructura los datos de **sociedades mercantiles constituidas** en España a través de la API pública TEMPUS del INE (tabla 13913). 

Los datos se desglosan por **territorio**, **tipo societario** (S.A., S.L., S. Comanditarias y Colectivas, y su agregado Mercantiles) y **periodo mensual**, obteniendo tanto el número de sociedades como el capital suscrito (en euros).

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/13913`.
2. **Extracción y desanidado** — Recorrido de la respuesta JSON para poblar un diccionario con las columnas: `id_const`, `territorio`, `id_tiempo`, `tipo`, `numero_sociedades` y `capital`.
3. **Limpieza en línea** — Corrección de escalas (FK_Escala 4 → multiplicar por 1000) y concatenación de filas de número de sociedades y capital en un mismo registro.
4. **Exportación** — Volcado a CSV en `../files/data_raw/empresas_constituidas.csv`.

### Contexto del proyecto
Estos datos se integran en un análisis de **resiliencia empresarial en España**, donde se cruzarán con disoluciones de empresas e IPC para estudiar el impacto del ciclo económico en la creación de empresas.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de conexión a la API
from src.api import connection_api
from src.api.config import API_URLS

In [5]:
url_const = API_URLS["constituidas"]  #Constituidas

In [6]:
data_const = connection_api.llamada_api(url_const)

In [7]:
data_const

[{'COD': 'SM21483',
  'Nombre': 'Sociedades Constituídas. Número de Sociedades. Total Nacional. Mercantiles. ',
  'FK_Unidad': 97,
  'FK_Escala': 1,
  'Data': [{'Fecha': 1777586400000,
    'FK_TipoDato': 2,
    'FK_Periodo': 5,
    'Anyo': 2026,
    'Valor': 11194.0,
    'Secreto': False},
   {'Fecha': 1774994400000,
    'FK_TipoDato': 2,
    'FK_Periodo': 4,
    'Anyo': 2026,
    'Valor': 11558.0,
    'Secreto': False},
   {'Fecha': 1772319600000,
    'FK_TipoDato': 2,
    'FK_Periodo': 3,
    'Anyo': 2026,
    'Valor': 14307.0,
    'Secreto': False},
   {'Fecha': 1769900400000,
    'FK_TipoDato': 2,
    'FK_Periodo': 2,
    'Anyo': 2026,
    'Valor': 12684.0,
    'Secreto': False},
   {'Fecha': 1767222000000,
    'FK_TipoDato': 2,
    'FK_Periodo': 1,
    'Anyo': 2026,
    'Valor': 11623.0,
    'Secreto': False},
   {'Fecha': 1764543600000,
    'FK_TipoDato': 2,
    'FK_Periodo': 12,
    'Anyo': 2025,
    'Valor': 11719.0,
    'Secreto': False},
   {'Fecha': 1761951600000,
    'FK_Ti

In [8]:
''' for dato in data_const:
    for serie in dato['Data']:
        print(f'{dato['Nombre']}, {serie['FK_TipoDato']}, {serie['FK_Periodo']}, {serie['Anyo']}, {serie['Valor']}')'''

" for dato in data_const:\n    for serie in dato['Data']:\n        print(f'{dato['Nombre']}, {serie['FK_TipoDato']}, {serie['FK_Periodo']}, {serie['Anyo']}, {serie['Valor']}')"

In [9]:
empresas_constituidas = { 
    'id_const': [], 
    'territorio': [], 
    'id_tiempo': [], 
    'tipo': [], 
    'numero_sociedades': [], 
    'capital': []            
}
contador = 1

for serie in data_const:
    nombre_completo = serie['Nombre']
    
    if "nacional" not in nombre_completo.lower():
        partes = nombre_completo.split('.', 3)
        territorio = partes[0].strip()
        tipo = partes[2].strip()
        unidad = partes[3].replace('.', '').strip() 

        # CORRECCIÓN EN LÍNEA: Si el tipo se quedó cortado en "S", le ponemos el nombre completo
        if tipo == "S":
            tipo = "S. Comanditarias y S. Colectivas"

        for data in serie['Data']:
            valor = data['Valor'] * 1000 if serie.get("FK_Escala") == 4 else data['Valor']
            id_tiempo = str(data['Anyo']) + str(data['FK_Periodo']).zfill(2)

            # COMPROBACIÓN: ¿Ya hemos añadido esta misma combinación?
            existe = False
            for i in range(len(empresas_constituidas['territorio'])):
                if (empresas_constituidas['territorio'][i] == territorio and 
                    empresas_constituidas['id_tiempo'][i] == id_tiempo and 
                    empresas_constituidas['tipo'][i] == tipo):
                    
                    if "capital" in unidad.lower():
                        empresas_constituidas['capital'][i] = int(valor)
                    else:
                        empresas_constituidas['numero_sociedades'][i] = int(valor)
                    existe = True
                    break
            
            # Si NO existe, creamos una nueva fila
            if not existe:
                empresas_constituidas['id_const'].append(contador)
                empresas_constituidas['territorio'].append(territorio)
                empresas_constituidas['id_tiempo'].append(id_tiempo)
                empresas_constituidas['tipo'].append(tipo)
                
                if "capital" in unidad.lower():
                    empresas_constituidas['capital'].append(int(valor))
                    empresas_constituidas['numero_sociedades'].append(None)
                else:
                    empresas_constituidas['capital'].append(None)
                    empresas_constituidas['numero_sociedades'].append(int(valor))
                
                contador += 1

In [10]:
empresas_constituidas

{'id_const': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  15

In [11]:
pd.DataFrame(empresas_constituidas).to_csv('../files/data_raw/empresas_constituidas.csv', index=False)